# 🚀 Financial Recommendation Model with UltraGCN (MongoDB Atlas)
## Mô hình Gợi ý Sản phẩm Tài chính & Ngân hàng Cá nhân hóa sử dụng UltraGCN

Notebook này nạp dữ liệu trực tiếp từ **MongoDB Atlas (`RinRec_DB`)** và triển khai quy trình huấn luyện **UltraGCN (Ultra Simplification Graph Convolutional Networks)** trên tập dữ liệu tương tác người dùng - sản phẩm tài chính ngân hàng, xuất ra và đồng bộ 2 tập dữ liệu:
1. `purchase_history.csv` / MongoDB collection `purchase_history`: Lịch sử giao dịch & dịch vụ quầy của các khách hàng được chọn.
2. `recommendations.csv` / MongoDB collection `recommendations`: Danh sách Top-5 sản phẩm gợi ý kèm điểm tương đồng, giá trị cốt lõi và kịch bản tư vấn cho Giao dịch viên (GDV).

### 1. Import các thư viện cần thiết & Thiết lập môi trường

In [11]:
import os
import sys
import random
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder

# Thiết lập đường dẫn import mongo_connector
for path in [os.path.abspath('..'), os.path.abspath('.')]:
    if path not in sys.path:
        sys.path.insert(0, path)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'⚡ Thiết bị tính toán: {device}')

⚡ Thiết bị tính toán: cpu


### 2. Nạp dữ liệu trực tiếp từ MongoDB Atlas (RinRec_DB)

In [12]:
import sys
from pathlib import Path

core_dir = Path(r"E:\RinRec_Demo\core")
if str(core_dir) not in sys.path:
    sys.path.insert(0, str(core_dir))

from mongo_connector import get_collection_df, save_df_to_collection

print('📡 Đang tải dữ liệu từ MongoDB Atlas (RinRec_DB)...')
df_customers = get_collection_df('dim_customer')
df_products = get_collection_df('DanhMucSanPham')
df_services = get_collection_df('DanhMucDichVu')
df_transactions = get_collection_df('factTransaction')
df_holdings = get_collection_df('factCustomerProduct')
df_rules = get_collection_df('RuleGoiY')

if df_customers.empty or df_transactions.empty:
    raise RuntimeError('❌ Không thể nạp dữ liệu từ MongoDB Atlas! Vui lòng kiểm tra kết nối mạng trong mongo_connector.py.')

print(f'✅ Nạp thành công: {len(df_customers)} Khách hàng | {len(df_products)} Sản phẩm | {len(df_transactions)} Giao dịch')

📡 Đang tải dữ liệu từ MongoDB Atlas (RinRec_DB)...
✅ Nạp thành công: 120 Khách hàng | 11 Sản phẩm | 3001 Giao dịch


### 3. Chọn ngẫu nhiên 50 Khách hàng & Xuất lịch sử giao dịch (purchase_history.csv)

In [13]:
valid_cids = sorted(df_customers['customer_id'].unique())
selected_cids = random.sample(valid_cids, min(50, len(valid_cids)))

hist_rows = []
for cid in selected_cids:
    cust = df_customers[df_customers['customer_id'] == cid].iloc[0]
    c_txns = df_transactions[df_transactions['customer_id'] == cid]
    for _, tx in c_txns.iterrows():
        hist_rows.append({
            'reviewerID': f'CUST_{cid:04d}',
            'reviewerName': cust['full_name'],
            'segment': cust['segment'],
            'category': tx['service_group'],
            'title': tx['service_name'],
            'brand': 'VPBank Financial',
            'price': f"{tx['amount']:,.0f} VND",
            'channel': tx['channel'],
            'transaction_time': tx['transaction_datetime']
        })

df_purchase_hist = pd.DataFrame(hist_rows)
import os

ROOT_DIR = r"E:\RinRec_Demo"
output_dir = os.path.join(ROOT_DIR, 'Product_Demo')
os.makedirs(output_dir, exist_ok=True)

out_ph_path = os.path.join(output_dir, 'purchase_history.csv')
out_ph_path = os.path.join(ROOT_DIR, 'Product_Demo', 'purchase_history.csv') if os.path.exists(os.path.join(ROOT_DIR, 'Product_Demo')) else 'purchase_history.csv'
df_purchase_hist.to_csv(out_ph_path, index=False, encoding='utf-8-sig')
save_df_to_collection('purchase_history', df_purchase_hist)
print(f'✅ Đã lưu {len(df_purchase_hist)} dòng vào MongoDB collection purchase_history và file {out_ph_path}.')
try:
    display(df_purchase_hist.head())
except Exception:
    print(df_purchase_hist.head())

✅ Đã lưu 1252 bản ghi vào collection 'purchase_history' trên MongoDB.
✅ Đã lưu 1252 dòng vào MongoDB collection purchase_history và file E:\RinRec_Demo\Product_Demo\purchase_history.csv.


,reviewerID,reviewerName,segment,category,title,brand,price,channel,transaction_time
0,CUST_0082,Lê Hữu Khánh,MASS,I. Ngân hàng số & Hỗ trợ,Thuê két sắt / dịch vụ giữ hộ tài sản,VPBank Financial,"433,242 VND",EBANK,2026-05-10 10:53:00
1,CUST_0082,Lê Hữu Khánh,MASS,J. Khách hàng doanh nghiệp,"Mở L/C và thanh toán quốc tế (T/T, D/P, D/A)",VPBank Financial,"5,059,998 VND",APP,2026-04-07 11:58:00
2,CUST_0082,Lê Hữu Khánh,MASS,J. Khách hàng doanh nghiệp,Đăng ký dịch vụ thu hộ qua mã VietQR / Virtual...,VPBank Financial,"17,797,299 VND",APP,2026-07-08 08:19:00
3,CUST_0082,Lê Hữu Khánh,MASS,C. Tiết kiệm & Tiền gửi,Cầm cố sổ tiết kiệm để vay vốn,VPBank Financial,"127,020,798 VND",COUNTER,2026-06-29 09:21:00
4,CUST_0082,Lê Hữu Khánh,MASS,I. Ngân hàng số & Hỗ trợ,Reset mật khẩu / cấp lại eToken / đổi thiết bị...,VPBank Financial,"4,936,380 VND",APP,2026-07-01 12:19:00


### 4. Xây dựng Đồ thị Bipartite Graph & Trọng số chuẩn hóa bậc node UltraGCN

In [14]:
service_to_prod = {
    'A. Tài khoản & Thông tin KH': 'SP024',
    'B. Giao dịch tiền mặt': 'SP004',
    'C. Tiết kiệm & Tiền gửi': 'SP001',
    'D. Thẻ': 'SP004',
    'E. Chuyển tiền & Thanh toán': 'SP021',
    'F. Ngoại tệ': 'SP018',
    'G. Tín dụng': 'SP008',
    'H. Bảo hiểm & Đầu tư': 'SP013',
    'I. Ngân hàng số & Hỗ trợ': 'SP021',
    'J. Khách hàng doanh nghiệp': 'SP025'
}

interactions = []
for _, row in df_holdings.iterrows():
    try:
        c_id = int(row['customer_id'])
        p_id = str(row['product_id'])
        bal = float(row.get('current_balance', 10_000_000))
        rating = 4.5 + min(0.5, np.log10(max(1, bal)) / 10.0)
        interactions.append({'customer_id': c_id, 'product_id': p_id, 'rating': rating})
    except Exception:
        continue

for _, row in df_transactions.iterrows():
    try:
        c_id = int(row['customer_id'])
        s_grp = str(row['service_group'])
        p_id = service_to_prod.get(s_grp, 'SP001')
        amt = float(row.get('amount', 1_000_000))
        rating = min(4.5, 1.0 + (np.log1p(amt) / 18.0) * 3.5)
        interactions.append({'customer_id': c_id, 'product_id': p_id, 'rating': rating})
    except Exception:
        continue

df_inter = pd.DataFrame(interactions)
df_grouped = df_inter.groupby(['customer_id', 'product_id'])['rating'].mean().reset_index()

user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
df_grouped['user_id'] = user_encoder.fit_transform(df_grouped['customer_id'])
df_grouped['item_id'] = item_encoder.fit_transform(df_grouped['product_id'])

num_users = len(user_encoder.classes_)
num_items = len(item_encoder.classes_)
num_nodes = num_users + num_items

user_freq = defaultdict(int)
item_freq = defaultdict(int)
for row in df_grouped.itertuples():
    user_freq[row.user_id] += 1
    item_key = row.item_idx if hasattr(row, 'item_idx') else row.item_id
    item_freq[item_key] = item_freq.get(item_key, 0) + 1

edge_index = []
edge_weight = []
for row in df_grouped.itertuples():
    u, i = row.user_id, row.item_id
    edge_index.append([u, num_users + i])
    edge_index.append([num_users + i, u])
    w = 1.0 / ((max(1, user_freq[u]) ** 0.5) * (max(1, item_freq[i]) ** 0.5))
    edge_weight.append(w)
    edge_weight.append(w)

edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
edge_weight = torch.tensor(edge_weight, dtype=torch.float32)

print(f'✅ Đồ thị Bipartite: {num_nodes} nodes, {edge_index.shape[1]} edges.')

✅ Đồ thị Bipartite: 145 nodes, 2154 edges.


### 5. Xây dựng và Huấn luyện mô hình UltraGCN

In [15]:
class UltraGCN(nn.Module):
    def __init__(self, num_nodes, emb_dim=32):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, emb_dim)
        nn.init.xavier_uniform_(self.emb.weight)

    def forward(self, edge_index, edge_weight):
        x = self.emb.weight
        row, col = edge_index
        norm = edge_weight
        out = torch.zeros_like(x)
        out.index_add_(0, row, x[col] * norm.unsqueeze(1))
        return out

embedding_dim = 32
model_ultra = UltraGCN(num_nodes, embedding_dim)
optimizer = torch.optim.Adam(model_ultra.parameters(), lr=0.005)
loss_fn = nn.MSELoss()

train_u = torch.tensor(df_grouped['user_id'].values, dtype=torch.long)
train_i = torch.tensor(df_grouped['item_id'].values, dtype=torch.long)
train_r = torch.tensor(df_grouped['rating'].values, dtype=torch.float32)

for epoch in range(15):
    model_ultra.train()
    optimizer.zero_grad()
    emb = model_ultra(edge_index, edge_weight)
    u_e = emb[train_u]
    i_e = emb[num_users + train_i]
    preds = (u_e * i_e).sum(dim=-1)
    loss = loss_fn(preds, train_r) + 1e-4 * torch.norm(emb)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:02d}/15: Loss = {loss.item():.4f}')

print('✅ Đã huấn luyện xong UltraGCN.')

Epoch 01/15: Loss = 19.4368
Epoch 05/15: Loss = 19.1990
Epoch 10/15: Loss = 18.6034
Epoch 15/15: Loss = 17.5743
✅ Đã huấn luyện xong UltraGCN.


### 6. Xuất danh sách Top-5 Đề xuất cho 50 Khách hàng (recommendations.csv & MongoDB)

In [16]:
model_ultra.eval()
emb = model_ultra(edge_index, edge_weight).detach().cpu().numpy()
u_embs = emb[:num_users]
i_embs = emb[num_users:]

rec_rows = []
for cid in selected_cids:
    cust = df_customers[df_customers['customer_id'] == cid].iloc[0]
    if cid in user_encoder.classes_:
        u_idx = user_encoder.transform([cid])[0]
        scores = u_embs[u_idx] @ i_embs.T
        top5_idx = np.argsort(scores)[-5:][::-1]
    else:
        top5_idx = range(5)
        
    casa = float(cust.get('casa_balance', 0))
    seg = str(cust.get('segment', 'MASS'))
    
    for rank, i_idx in enumerate(top5_idx, 1):
        p_id = item_encoder.inverse_transform([i_idx])[0]
        p_match = df_products[df_products['Ma_SP'] == p_id]
        if p_match.empty:
            p_name = f'Sản phẩm {p_id}'
            p_nhom = 'Tài chính'
            p_val = 'Giải pháp tài chính'
            p_fee = 'Theo biểu phí'
            p_min = 0
        else:
            p_info = p_match.iloc[0]
            p_name = p_info['Ten_san_pham']
            p_nhom = p_info['Nhom']
            p_val = p_info['Gia_tri_cot_loi']
            p_fee = p_info.get('Lai_suat_Phi', 'Theo biểu phí')
            p_min = float(p_info.get('So_tien_toi_thieu', 0))
        
        match_score = int(88 + (5 - rank) * 2 + random.randint(-1, 1))
        match_score = min(99, max(75, match_score))
        
        if 'Tiết kiệm' in str(p_nhom):
            gdv_script = f'Khách có {casa:,.0f}đ số dư nhàn rỗi. Gợi ý {p_name} để hưởng lãi suất ưu đãi.'
        elif 'Thẻ' in str(p_nhom):
            gdv_script = f'Khách có dòng tiền đều đặn. Gợi ý mở {p_name} miễn lãi 45 ngày và hoàn tiền chi tiêu.'
        elif 'Bảo hiểm' in str(p_nhom):
            gdv_script = f'Gợi ý {p_name} để bảo vệ tài chính toàn diện gia đình.'
        elif 'Vay' in str(p_nhom):
            gdv_script = f'Tư vấn gói {p_name} với lãi suất ưu đãi cố định.'
        else:
            gdv_script = f'Gói {p_name}: {p_val}'
            
        rec_rows.append({
            'reviewerID': f'CUST_{cid:04d}',
            'reviewerName': cust['full_name'],
            'segment': seg,
            'category': p_nhom,
            'title': p_name,
            'brand': 'VPBank Financial',
            'price': f"{p_min:,.0f} VND",
            'rate_or_fee': p_fee,
            'value_proposition': p_val,
            'match_score': f'{match_score}%',
            'gdv_script': gdv_script
        })

df_recs = pd.DataFrame(rec_rows)
out_rc_path = os.path.join(ROOT_DIR, 'Product_Demo', 'recommendations.csv') if os.path.exists(os.path.join(ROOT_DIR, 'Product_Demo')) else 'recommendations.csv'
df_recs.to_csv(out_rc_path, index=False, encoding='utf-8-sig')
save_df_to_collection('recommendations', df_recs)
print(f'✅ Đã xuất {len(df_recs)} dòng vào MongoDB collection recommendations và file {out_rc_path}.')
try:
    display(df_recs.head(10))
except Exception:
    print(df_recs.head(10))

✅ Đã lưu 250 bản ghi vào collection 'recommendations' trên MongoDB.
✅ Đã xuất 250 dòng vào MongoDB collection recommendations và file E:\RinRec_Demo\Product_Demo\recommendations.csv.


,reviewerID,reviewerName,segment,category,title,brand,price,rate_or_fee,value_proposition,match_score,gdv_script
0,CUST_0082,Lê Hữu Khánh,MASS,Tài chính,Sản phẩm SP021,VPBank Financial,0 VND,Theo biểu phí,Giải pháp tài chính,97%,Gói Sản phẩm SP021: Giải pháp tài chính
1,CUST_0082,Lê Hữu Khánh,MASS,Thẻ,Thẻ ghi nợ quốc tế,VPBank Financial,0 VND,Phí phát hành + thường niên,Thanh toán online và mua sắm nước ngoài an toà...,94%,Khách có dòng tiền đều đặn. Gợi ý mở Thẻ ghi n...
2,CUST_0082,Lê Hữu Khánh,MASS,Vay,Vay mua ô tô,VPBank Financial,"100,000,000 VND",Theo chương trình từng thời kỳ,"Chỉ cần trả trước một phần là có xe, thế chấp ...",93%,Tư vấn gói Vay mua ô tô với lãi suất ưu đãi cố...
3,CUST_0082,Lê Hữu Khánh,MASS,Tài chính,Sản phẩm SP025,VPBank Financial,0 VND,Theo biểu phí,Giải pháp tài chính,89%,Gói Sản phẩm SP025: Giải pháp tài chính
4,CUST_0082,Lê Hữu Khánh,MASS,Tài chính,Sản phẩm SP024,VPBank Financial,0 VND,Theo biểu phí,Giải pháp tài chính,88%,Gói Sản phẩm SP024: Giải pháp tài chính
5,CUST_0015,Bùi Kim Em,MASS,Tài chính,Sản phẩm SP021,VPBank Financial,0 VND,Theo biểu phí,Giải pháp tài chính,95%,Gói Sản phẩm SP021: Giải pháp tài chính
6,CUST_0015,Bùi Kim Em,MASS,Thẻ,Thẻ ghi nợ quốc tế,VPBank Financial,0 VND,Phí phát hành + thường niên,Thanh toán online và mua sắm nước ngoài an toà...,95%,Khách có dòng tiền đều đặn. Gợi ý mở Thẻ ghi n...
7,CUST_0015,Bùi Kim Em,MASS,Vay,Vay mua ô tô,VPBank Financial,"100,000,000 VND",Theo chương trình từng thời kỳ,"Chỉ cần trả trước một phần là có xe, thế chấp ...",92%,Tư vấn gói Vay mua ô tô với lãi suất ưu đãi cố...
8,CUST_0015,Bùi Kim Em,MASS,Tài chính,Sản phẩm SP025,VPBank Financial,0 VND,Theo biểu phí,Giải pháp tài chính,91%,Gói Sản phẩm SP025: Giải pháp tài chính
9,CUST_0015,Bùi Kim Em,MASS,Tài chính,Sản phẩm SP024,VPBank Financial,0 VND,Theo biểu phí,Giải pháp tài chính,89%,Gói Sản phẩm SP024: Giải pháp tài chính
